# 03 ML Pipeline: Customer Tribe Discovery

This notebook starts from the prepared parquet outputs created by Notebook 02. It does not repeat raw loading, cleaning, or exploratory data quality work from Notebooks 01 and 02.

The goal is to discover product-first customer tribes from purchase behavior. The client hypothesis is roughly 10-15 tribes, but that range is not used as a modeling constraint. The final recommendation is selected from metrics, stability-ready diagnostics, cluster balance, product/sector lift interpretability, and business usefulness.

## Table of Contents

| Section | What it covers |
|---|---|
| [Stage 0: Load Prepared Data](#stage-0) | Sets the run mode, validates paths, loads prepared transactions, and previews the input data. |
| [Stage 0.5: Expensive Cache Audit](#stage-0-5) | Checks whether Stage 1-6 artifacts can be reused safely from cache. |
| [Stage 1: Basket Construction](#stage-1) | Converts checkout records into basket sentences for product embedding training. |
| [Stage 2: Item2Vec Product Embeddings](#stage-2) | Builds or loads product embeddings from product co-purchase behavior. |
| [Stage 3: Product Embedding Validation](#stage-3) | Reviews product-neighbor diagnostics and embedding quality checks. |
| [Stage 4: Customer Embeddings](#stage-4) | Creates product-first customer embeddings from purchased products and quantities. |
| [Stage 5: Official Feature Set](#stage-5) | Builds the official product/quantity feature set and supporting non-demographic behavior summaries. |
| [Stage 6: Hard UMAP-HDBSCAN Core Tribe Discovery](#stage-6) | Builds the UMAP customer manifold, runs hard HDBSCAN, and exports core-tribe diagnostics. |
| [Stage 6.4: Cluster Stability and Profile Readiness](#stage-6-4) | Tests cluster-level stability, confidence, and readiness before profiling. |
| [Stage 7: Deep Tribe Profiling and Interpretation](#stage-7) | Turns the selected clustering solution into a structured evidence storyline with linked supporting proof. |

Tribe discovery is documented as a clear late-stage progression:

| Layer | Pipeline stage | What it proves | Main outputs |
|---|---|---|---|
| 1. Product-only modeling signal | Stages 4-5 | Official clustering uses product identity and quantity, not demographics, spend, or behavior KPIs. | Customer embeddings, feature-set diagnostics |
| 2. Organic core tribes | Stage 6.1-6.4 | Dense product-purchase behavior groups exist without forcing every customer into a tribe, and the retained clusters are checked before profiling. | UMAP representation, hard HDBSCAN assignments, representation/density evidence, stability/readiness report |
| 3. Evidence storyline | Stage 7 | The selected solution is read through an ordered ladder: readiness, product distinctiveness, supporting context, subsegments, and caveats. | Evidence storyline, profile readiness table, all-tribe comparison, evidence dashboard |
| 4. Supporting proof and optional synthesis | Stage 7 | Detailed artifacts support specific claims without crowding the main read; aggregate evidence can also be packaged for constrained LLM review. | Product tables, dossier, clustering atlas, LLM prompt pack |

Read the official result in that order: first whether hard organic core tribes exist, then whether Stage 6.4 stability/readiness supports profiling them, then whether the Stage 7 evidence storyline makes them interpretable, distinct, caveated, and profileable. Spend and KPIs are interpretation context only; they are never part of the clustering signal.


<a id="stage-0"></a>

## Stage 0: Load Prepared Data

Notebook 03 consumes `df_combined.parquet`, validates the fields required for ML, and merges product metadata only if the prepared file does not already contain it.

In [ ]:
import os
from pathlib import Path
import sys

from IPython.display import Image, Markdown, display

# Optional notebook override. Set to "dev" or "prod" to force a mode;
# leave as None to honor CARREFOUR_MODE, then the YAML default_mode.
NOTEBOOK_MODE_OVERRIDE = "dev"

for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    if (candidate / "src").is_dir():
        project_root = candidate
        break
else:
    project_root = Path.cwd().resolve()

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.config import configure_mode, load_config
from src.cache_audit import assert_mode_path_audit
from src.data_loader import load_prepared_transactions, peek
from src.stage_reports import display_stage_report, write_stage_report
from src.utils import set_global_seed

RUN_MODE = (NOTEBOOK_MODE_OVERRIDE or os.environ.get("CARREFOUR_MODE") or load_config().mode).strip().lower()
if RUN_MODE not in {"dev", "prod"}:
    raise ValueError(f"RUN_MODE must be 'dev' or 'prod', got {RUN_MODE!r}")
os.environ["CARREFOUR_MODE"] = RUN_MODE

previous_mode = globals().get("MODE")
CONFIG = configure_mode(RUN_MODE)
MODE = CONFIG.mode
DATA_PROCESSED = CONFIG.data_processed
MODELS = CONFIG.models
OUTPUTS = CONFIG.outputs

if previous_mode and previous_mode != MODE:
    for stale_name in [
        "transactions",
        "basket_path",
        "item2vec_model",
        "product_embeddings_path",
        "embedding_validation_csv",
        "embedding_validation_detail_md",
        "customer_embeddings_path",
        "behavior_path",
        "feature_sets",
        "model_suite",
        "stage6_diagnostics",
        "cluster_stability",
        "stability_table",
        "cluster_readiness_table",
        "stage6_winner_key",
        "stage6_4_projection_dir",
        "stage6_4_projection_figures",
        "profile_paths",
        "selected_key",
        "selection_evidence_path",
        "selected",
        "selected_assignment_path",
        "selected_profile_path",
        "stage7_artifacts_dir",
        "stage7_figures_dir",
        "stage7_quality",
        "stage7_readiness_path",
        "stage7_readiness",
        "stage7_ready_clusters",
        "stage7_review_clusters",
        "product_summary_paths",
        "tribe_product_tables",
        "tribe_comparison_artifacts",
        "tribe_comparison",
        "customer_metric_tests_path",
        "customer_metric_tests",
        "stage7_significant_customer_metric_tests",
        "stage7_profile_overview",
        "stage7_profile_evidence",
        "stage7_noise_audit",
        "stage7_noise_audit_paths",
        "stage7_noise_summary_row",
        "stage7_storyline_paths",
        "stage7_storyline",
        "stage7_cluster_summary_paths",
        "tribe_vs_population_dashboard",
        "profile_comparison_heatmap",
        "stage7_theme_lift_heatmap",
        "stage7_lift_dir",
        "top_lift_paths",
        "profile_export",
        "subsegment_opportunities",
        "subsegment_opportunities_path",
        "customer_subsegment_paths",
        "discovered_term_paths",
        "campaign_signal_paths",
        "profile_report_path",
        "clustering_atlas_paths",
        "llm_prompt_paths",
    ]:
        globals().pop(stale_name, None)

set_global_seed(CONFIG.random_seed)
CONFIG.ensure_directories()
mode_path_audit = assert_mode_path_audit(CONFIG)

transactions = load_prepared_transactions(cfg=CONFIG)
print(f"Run mode: {MODE}")
print(f"Data path: {DATA_PROCESSED}")
print(f"Model path: {MODELS}")
print(f"Output path: {OUTPUTS}")
peek(transactions, 3)

from src.visualization import plot_prepared_data_overview

stage0_figure = plot_prepared_data_overview(transactions, cfg=CONFIG)
stage0_report = write_stage_report(
    "00",
    "Load Prepared Data",
    summary=[
        f"Run mode active: {MODE}",
        "Mode/path audit passed; dev and prod namespaces are not mixed.",
        "Prepared transactions loaded from the configured mode path.",
    ],
    metrics={
        "mode_path_checks": mode_path_audit.height,
        "failed_mode_path_checks": mode_path_audit.filter(mode_path_audit.get_column("status") == "fail").height,
    },
    figures={"Prepared data overview": stage0_figure},
    artifacts={"Prepared transactions": CONFIG.prepared_transactions_path},
    cfg=CONFIG,
)
display_stage_report(stage0_report)
display(mode_path_audit.select(["check", "status", "exists", "reason"]).head(12))
display(Image(filename=str(stage0_figure)))


<a id="stage-0-5"></a>

## Stage 0.5: Expensive Cache Audit

Before running the expensive pipeline stages, this table checks whether Stage 1-6 artifacts will be reused from cache. A cache hit requires the artifact to exist and its metadata hash to match the current input files and config.

In [ ]:
import importlib
import polars as pl

import src.utils
importlib.reload(src.utils)
import src.cache_audit
importlib.reload(src.cache_audit)
from src.cache_audit import stage_1_6_cache_audit
from src.stage_reports import display_stage_report, write_stage_report
# Optional: if existing artifacts were produced from the current data/config but metadata is missing,
# uncomment the next two lines once to adopt them into the cache manifest.
# from src.cache_audit import adopt_existing_stage_1_6_cache_metadata
# display(adopt_existing_stage_1_6_cache_metadata(CONFIG))


cache_audit = stage_1_6_cache_audit(CONFIG)
cache_misses = cache_audit.filter(~pl.col("cache_hit"))

stage0_5_report = write_stage_report(
    "00_5",
    "Expensive Cache Audit",
    summary=[
        "Checks whether expensive Stage 1-6 artifacts can be reused safely.",
        "Artifacts with cache_hit=False will be rebuilt when their stage runs.",
    ],
    metrics={
        "audited_artifacts": cache_audit.height,
        "cache_ready_artifacts": cache_audit.filter(pl.col("cache_hit")).height,
        "rebuild_artifacts": cache_misses.height,
    },
    artifacts={"Artifact metadata manifest": CONFIG.outputs / ".artifact_metadata.json"},
    cfg=CONFIG,
)
display_stage_report(stage0_5_report)
cache_display_cols = ["stage", "artifact", "cache_hit", "exists", "reason", "path"]
display(cache_audit.select([c for c in cache_display_cols if c in cache_audit.columns]).head(20))

if cache_misses.height:
    print("Artifacts listed as cache_hit=False will be rebuilt if their stage is run.")
else:
    print("All expensive Stage 1-6 artifacts are cache-ready.")


<a id="stage-1"></a>

## Stage 1: Basket Construction

Each ticket is treated as a basket sentence and each product id is a token. Products are not repeated by quantity unless `baskets.repeat_product_by_quantity` is enabled in the config.

In [ ]:
from IPython.display import Image, display
import polars as pl

from src.basket_builder import (
    basket_summary,
    build_basket_sentences,
    build_basket_staple_diagnostics,
)
from src.stage_reports import display_stage_report, write_stage_report
from src.visualization import (
    plot_basket_staple_diagnostics,
    plot_basket_summary,
)

basket_path = build_basket_sentences(transactions=transactions, cfg=CONFIG)
stage1_basket_summary = basket_summary(basket_path)
stage1_figure = plot_basket_summary(basket_path, cfg=CONFIG)

stage1_diagnostics = build_basket_staple_diagnostics(transactions=transactions, cfg=CONFIG)
stage1_staple_figure = plot_basket_staple_diagnostics(
    stage1_diagnostics["product_diagnostics"],
    stage1_diagnostics["basket_exposure"],
    cfg=CONFIG,
)
stage1_product_diagnostics = pl.scan_parquet(stage1_diagnostics["product_diagnostics"])
stage1_common_products = stage1_product_diagnostics.filter(pl.col("common_product_candidate")).select(pl.len()).collect()[0, 0]
stage1_common_preview_rows = (
    pl.read_csv(stage1_diagnostics["common_products_csv"]).height
    if stage1_diagnostics["common_products_csv"].exists()
    else 0
)
stage1_downsampling_diag_cfg = CONFIG.get("baskets.diagnostics", {}) or {}
stage1_downsampling_plan_path = CONFIG.artifacts / str(stage1_downsampling_diag_cfg.get("output_dir", "stage1")) / str(stage1_downsampling_diag_cfg.get("downsampling_plan_output", "common_product_downsampling_plan.parquet"))
stage1_downsampling_summary_path = CONFIG.artifacts / str(stage1_downsampling_diag_cfg.get("output_dir", "stage1")) / str(stage1_downsampling_diag_cfg.get("downsampling_summary_output", "basket_downsampling_summary.csv"))
stage1_metrics = stage1_basket_summary.row(0, named=True)
stage1_metrics["common_product_candidates"] = stage1_common_products
stage1_metrics["common_product_preview_rows"] = stage1_common_preview_rows
stage1_downsampling_summary = None
if stage1_downsampling_summary_path.exists():
    stage1_downsampling_summary = pl.read_csv(stage1_downsampling_summary_path)
    stage1_metrics.update(
        stage1_downsampling_summary.select(
            [
                "basket_retention_pct",
                "unique_pair_retention_pct",
                "fallback_basket_pct",
                "auto_excluded_products",
                "downsampled_common_candidate_products",
            ]
        ).row(0, named=True)
    )
stage1_report = write_stage_report(
    "01",
    "Basket Sentences and Common-Product Exposure",
    summary=[
        "Basket sentences use product identity; active Stage 1 settings do not repeat products by unidades.",
        "Common-product exposure and downsampling retention are audited before Item2Vec training.",
    ],
    metrics=stage1_metrics,
    figures={
        "Basket token summary": stage1_figure,
        "Common-product exposure": stage1_staple_figure,
    },
    artifacts={
        "Basket sentences": basket_path,
        "Common-product candidates CSV": stage1_diagnostics["common_products_csv"],
        "Product ubiquity diagnostics": stage1_diagnostics["product_diagnostics"],
        "Downsampling plan": stage1_downsampling_plan_path if stage1_downsampling_plan_path.exists() else None,
        "Downsampling summary": stage1_downsampling_summary_path if stage1_downsampling_summary_path.exists() else None,
    },
    cfg=CONFIG,
)
display_stage_report(stage1_report)
display(stage1_basket_summary)
if stage1_downsampling_summary is not None:
    display(stage1_downsampling_summary)
display(Image(filename=str(stage1_figure)))
display(Image(filename=str(stage1_staple_figure)))
print(f"Stage 2 Item2Vec basket path: {basket_path}")


<a id="stage-2"></a>

## Stage 2: Item2Vec Product Embeddings

The Word2Vec model learns product proximity from basket co-occurrence. These product embeddings are the core signal used to represent customers.

The training function prints the active hyperparameters and one progress line per epoch. If a cached model already exists, it prints the cache details instead; pass `force=True` to retrain.

In [ ]:
from IPython.display import Image
import polars as pl

from src.item2vec import save_product_embeddings, train_item2vec, write_item2vec_training_diagnostics
from src.stage_reports import display_stage_report, write_stage_report
from src.visualization import plot_product_embedding_diagnostics

item2vec_model = train_item2vec(basket_path, cfg=CONFIG, verbose=True)
stage2_training_diagnostics_path = write_item2vec_training_diagnostics(basket_path, cfg=CONFIG)
stage2_training_diagnostics = pl.read_csv(stage2_training_diagnostics_path)
product_embeddings_path = save_product_embeddings(item2vec_model, cfg=CONFIG)
stage2_figure = plot_product_embedding_diagnostics(product_embeddings_path, cfg=CONFIG)
stage2_schema = pl.read_parquet(product_embeddings_path, n_rows=1).columns
stage2_metrics = {
    "embedded_products": pl.scan_parquet(product_embeddings_path).select(pl.len()).collect()[0, 0],
    "embedding_dimensions": len([c for c in stage2_schema if c.startswith("emb_")]),
    "word2vec_window": CONFIG.get("word2vec.window"),
    "word2vec_min_count": CONFIG.get("word2vec.min_count"),
}
stage2_metrics.update(
    stage2_training_diagnostics.select(
        [
            "baskets",
            "training_baskets",
            "skipped_short_basket_pct",
            "capped_basket_pct",
            "training_token_retention_pct",
            "effective_window",
        ]
    ).row(0, named=True)
)
stage2_report = write_stage_report(
    "02",
    "Product Embedding Training",
    summary=[
        "Item2Vec is trained on Stage 1 product-token baskets after Stage 2 corpus limits.",
        "Product quantities enter the official customer vectors in Stage 4, not as repeated Stage 2 tokens under the active config.",
    ],
    metrics=stage2_metrics,
    figures={"Product embedding diagnostics": stage2_figure},
    artifacts={"Product embeddings": product_embeddings_path, "Training corpus diagnostics": stage2_training_diagnostics_path},
    cfg=CONFIG,
)
display_stage_report(stage2_report)
display(stage2_training_diagnostics)
display(Image(filename=str(stage2_figure)))


<a id="stage-3"></a>

## Stage 3: Product Embedding Validation

Before clustering customers, the nearest-neighbor report checks whether embeddings capture meaningful substitutes, complements, or shared basket missions.

In [ ]:
from IPython.display import Image
import polars as pl

from src.embedding_validation import product_embedding_guardrail_status, validate_product_embeddings
from src.stage_reports import display_stage_report, write_stage_report
from src.visualization import plot_embedding_validation_quality_extracts, plot_embedding_validation_summary

embedding_validation_csv, embedding_validation_detail_md = validate_product_embeddings(
    product_embeddings_path,
    transactions=transactions,
    cfg=CONFIG,
)
stage3_figure = plot_embedding_validation_summary(embedding_validation_csv, cfg=CONFIG)
stage3_extract_figures = {}
if CONFIG.get("embedding_validation.write_extract_figures", False):
    stage3_extract_figures = plot_embedding_validation_quality_extracts(embedding_validation_csv, cfg=CONFIG)
stage3_guardrail_status = product_embedding_guardrail_status(embedding_validation_csv, cfg=CONFIG)
stage3_figures = {"Embedding validation summary": stage3_figure}
stage3_figures.update({f"Quality extract: {name}": path for name, path in stage3_extract_figures.items()})
stage3_issues = stage3_guardrail_status["issues"] or ["None"]
stage3_report = write_stage_report(
    "03",
    "Product Embedding Validation",
    summary=[
        f"Guardrail status: {stage3_guardrail_status['status']}",
        stage3_guardrail_status["summary"],
        "Issues: " + "; ".join(stage3_issues[:4]),
    ],
    metrics={
        "validated_products": pl.read_csv(embedding_validation_csv).height,
        "guardrail_issue_count": 0 if stage3_guardrail_status["issues"] is None else len(stage3_guardrail_status["issues"]),
    },
    figures=stage3_figures,
    artifacts={
        "Validation CSV": embedding_validation_csv,
        "Hubness CSV": stage3_guardrail_status["hubness_csv"],
    },
    cfg=CONFIG,
)
display_stage_report(stage3_report)
display(Image(filename=str(stage3_figure)))
for figure_path in stage3_extract_figures.values():
    display(Image(filename=str(figure_path)))
stage3_guardrail_status


<a id="stage-4"></a>

## Stage 4: Customer Embeddings

Customer vectors are weighted means of product embeddings. The official weighting starts from product quantities (`unidades`), applies the configured product-purchase recency decay and repeated-basket frequency scaling, then aggregates product vectors. This keeps clustering driven by purchased products, purchase intensity, recency, and repeat behavior rather than spend. IDF-downweighted variants remain sandbox-only unless promoted into the official YAML recipe after evidence review.

In [ ]:
from IPython.display import Image, display
import polars as pl

from src.customer_embeddings import build_customer_embeddings
from src.stage_reports import display_stage_report, write_stage_report
from src.visualization import plot_customer_embedding_diagnostics

customer_embeddings_path = build_customer_embeddings(
    product_embeddings_path,
    transactions=transactions,
    normalize_vectors=CONFIG.get("customer_embeddings.normalize_vectors", False),
    cfg=CONFIG,
)
stage4_figure = plot_customer_embedding_diagnostics(customer_embeddings_path, cfg=CONFIG)

stage4_diag_cfg = CONFIG.get("customer_embeddings.diagnostics", {}) or {}
stage4_weight_summary_path = (
    CONFIG.artifacts
    / str(stage4_diag_cfg.get("output_dir", "stage4"))
    / f"{stage4_diag_cfg.get('output_prefix', 'customer_embedding')}_weight_diagnostics.csv"
)
stage4_display_cols = [
    "weight_strategy",
    "quantity_transform",
    "common_product_weight_share_pct",
    "mean_customer_common_product_weight_share_pct",
    "p95_customer_common_product_weight_share_pct",
    "top_product_weight_share_pct",
    "top_10_product_weight_share_pct",
    "mean_customer_top_product_weight_share_pct",
    "stage4_gate_status",
    "stage4_gate_issues",
    "recency_weighting_enabled",
    "recency_reference_date",
    "recency_half_life_days",
    "mean_recency_multiplier",
    "frequency_weighting_enabled",
    "frequency_transform",
    "mean_frequency_multiplier",
    "p95_customer_product_basket_count",
    "line_coverage_pct",
    "unit_coverage_pct",
    "product_coverage_pct",
    "customer_coverage_pct",
    "zero_embedded_customer_pct",
]
stage4_weight_summary = None
stage4_report_metrics = {
    "embedded_customers": pl.scan_parquet(customer_embeddings_path).select(pl.len()).collect()[0, 0]
}
if stage4_weight_summary_path.exists():
    stage4_weight_summary = pl.read_csv(stage4_weight_summary_path)
    available_stage4_cols = [c for c in stage4_display_cols if c in stage4_weight_summary.columns]
    if available_stage4_cols:
        stage4_report_metrics.update(stage4_weight_summary.select(available_stage4_cols).row(0, named=True))
stage4_report = write_stage_report(
    "04",
    "Customer Embeddings Coverage and Dominance Gates",
    summary=[
        "Customer vectors are weighted aggregations of purchased-product embeddings.",
        "Coverage gates check how much transaction signal survives product embedding coverage.",
        "Dominance gates check whether common products overwhelm customer vectors.",
    ],
    metrics=stage4_report_metrics,
    figures={"Customer embedding diagnostics": stage4_figure},
    artifacts={
        "Customer embeddings": customer_embeddings_path,
        "Weight diagnostics": stage4_weight_summary_path if stage4_weight_summary_path.exists() else None,
    },
    cfg=CONFIG,
    max_metric_rows=32,
)
display_stage_report(stage4_report)
display(Image(filename=str(stage4_figure)))
if stage4_weight_summary is not None:
    display(stage4_weight_summary.select([c for c in stage4_display_cols if c in stage4_weight_summary.columns]))


<a id="stage-5"></a>

## Stage 5: Official Feature Set

Behavioral features are stored separately for post-clustering profiling and business interpretation. The official modeling feature set remains the product/quantity customer embedding.

In [ ]:
from IPython.display import Image
import polars as pl

from src.feature_engineering import build_behavioral_features, build_feature_set, build_feature_set_diagnostics
from src.stage_reports import display_stage_report, write_stage_report
from src.visualization import plot_behavioral_feature_summary, plot_feature_set_summary

behavior_path = build_behavioral_features(transactions=transactions, cfg=CONFIG)
official_feature_set = build_feature_set(customer_embeddings_path, variant="embeddings_only", cfg=CONFIG)
feature_sets = {
    "embeddings_only": official_feature_set,
}
stage5_behavior_figure = plot_behavioral_feature_summary(behavior_path, cfg=CONFIG)
stage5_feature_set_figure = plot_feature_set_summary(feature_sets, cfg=CONFIG)
stage5_feature_diagnostics_path = build_feature_set_diagnostics(feature_sets, cfg=CONFIG)
stage5_feature_diagnostics = pl.read_csv(stage5_feature_diagnostics_path)
stage5_diagnostic_display_cols = [
    "feature_set_name",
    "is_selection_feature_set",
    "rows",
    "feature_count",
    "finite_pct",
    "null_pct",
    "zero_variance_feature_count",
    "customer_alignment_status",
    "missing_vs_baseline_customers",
    "extra_vs_baseline_customers",
    "neighbor_overlap_vs_baseline_pct",
    "selection_warning",
]
stage5_report = write_stage_report(
    "05",
    "Official Product/Quantity Feature Set",
    summary=[
        "Official tribe selection uses embeddings_only to keep clustering product-first.",
        "Behavioral features are prepared separately for post-clustering interpretation.",
    ],
    metrics={
        "feature_sets_built": len(feature_sets),
        "selection_feature_set": CONFIG.get("modeling.feature_set_for_selection", "embeddings_only"),
        "feature_diagnostics_rows": stage5_feature_diagnostics.height,
    },
    figures={
        "Behavioral feature summary": stage5_behavior_figure,
        "Feature set summary": stage5_feature_set_figure,
    },
    artifacts={"Behavioral features": behavior_path, "Feature diagnostics": stage5_feature_diagnostics_path, **feature_sets},
    cfg=CONFIG,
)
display_stage_report(stage5_report)
display(Image(filename=str(stage5_behavior_figure)))
display(Image(filename=str(stage5_feature_set_figure)))
display(stage5_feature_diagnostics.select([c for c in stage5_diagnostic_display_cols if c in stage5_feature_diagnostics.columns]))


<a id="stage-6"></a>

## Stage 6: Hard UMAP-HDBSCAN Core Tribe Discovery

This is the official organic-tribe discovery step. It uses the selected product/quantity customer feature set, builds a UMAP representation for clustering, and then runs hard HDBSCAN with noise retained. Customers labeled `-1` are not forced into a tribe; they are the honest non-core population for this recipe.

- Stage 6.1 builds the UMAP customer manifold from `embeddings_only`.
- Stage 6.2 runs HDBSCAN without soft assignment, so discovered tribes are dense organic cores.
- Stage 6.3 writes evidence for UMAP neighborhood retention, HDBSCAN density validity, noise, balance, and assignment provenance.
- Stage 6.4 checks cluster-level stability, confidence, and profile readiness before Stage 7 interpretation.

UMAP is treated as a clustering representation, not as automatic proof. In prod, the expensive UMAP/HDBSCAN fitting steps use the configured deterministic customer sample and then transform/assign the full customer population in batches. The final claim is only credible when Stage 6 density structure, Stage 6.4 stability/readiness, and Stage 7 product-lift profiles agree.

In [ ]:
import polars as pl
from IPython.display import Image, display

from src.model_selection import (
    build_candidate_model_diagnostics,
    build_official_umap_core_representation,
    build_stage6_hdbscan_diagnostics,
    build_stage6_representation_cluster_diagnostics,
    build_stage6_umap_diagnostics,
    model_suite_from_single_candidate,
    run_official_hdbscan_core,
)
from src.stage_reports import display_stage_report, write_stage_report
from src.visualization import (
    plot_stage6_hdbscan_assignment_map,
    plot_stage6_quality_evidence,
    plot_stage6_umap_representation,
)

selection_feature_set = CONFIG.get("modeling.feature_set_for_selection", "embeddings_only")
if "feature_sets" not in globals():
    feature_set_outputs = CONFIG.get("feature_sets.outputs", {})
    selection_feature_filename = feature_set_outputs.get(selection_feature_set, f"feature_set_{selection_feature_set}.parquet")
    feature_sets = {
        selection_feature_set: CONFIG.outputs / "features" / selection_feature_filename,
    }
    missing_feature_sets = [path for path in feature_sets.values() if not path.exists()]
    if missing_feature_sets:
        raise FileNotFoundError(
            "Stage 6 needs the Stage 5 feature-set parquet files. "
            f"Missing: {missing_feature_sets}. Run Stage 5 first."
        )

if selection_feature_set != "embeddings_only":
    raise ValueError(
        "Official Stage 6 core tribe discovery expects modeling.feature_set_for_selection='embeddings_only'. "
        "Update the official modeling contract before running behavior-enhanced feature sets here."
    )
stage6_feature_path = feature_sets[selection_feature_set]
stage6_feature_path


### Stage 6.1: UMAP Representation Check

Build the official UMAP customer manifold from the product/quantity feature set, then stop if row alignment, dimensionality, nulls, or finite-value checks fail.

In [ ]:
stage6_umap_path = build_official_umap_core_representation(stage6_feature_path, cfg=CONFIG)
stage6_umap_checks_path = build_stage6_umap_diagnostics(stage6_feature_path, stage6_umap_path, cfg=CONFIG)
stage6_umap_checks = pl.read_csv(stage6_umap_checks_path)
stage6_umap_figure = plot_stage6_umap_representation(stage6_umap_path, cfg=CONFIG)

display(stage6_umap_checks)
display(Image(filename=str(stage6_umap_figure)))
if stage6_umap_checks[0, "check_status"] != "pass":
    raise ValueError(f"Stage 6.1 UMAP checks failed: {stage6_umap_checks[0, 'check_issues']}")


### Stage 6.2: Hard HDBSCAN Core Assignment Check

Run HDBSCAN on the UMAP representation with soft assignment disabled. Noise remains `tribe_id = -1`; this cell stops before Stage 6.3 only for structural issues such as row misalignment, duplicate assignments, or accidental soft assignment. Quality-gate failures such as high noise are carried into Stage 6.3 and Stage 6.4 for diagnosis.

In [ ]:
stage6_assignment_path, stage6_result_path, stage6_core_candidate = run_official_hdbscan_core(stage6_umap_path, cfg=CONFIG)
stage6_hdbscan_checks_path = build_stage6_hdbscan_diagnostics(
    stage6_umap_path,
    stage6_assignment_path,
    stage6_result_path,
    cfg=CONFIG,
)
stage6_hdbscan_checks = pl.read_csv(stage6_hdbscan_checks_path)
stage6_assignment_source_table = (
    pl.read_parquet(stage6_assignment_path)
    .group_by("assignment_source")
    .agg(pl.len().alias("customers"))
    .sort("customers", descending=True)
)
stage6_assignment_figure = plot_stage6_hdbscan_assignment_map(stage6_umap_path, stage6_assignment_path, cfg=CONFIG)

display(stage6_hdbscan_checks)
display(stage6_assignment_source_table)
display(Image(filename=str(stage6_assignment_figure)))
blocking_status_col = "blocking_check_status" if "blocking_check_status" in stage6_hdbscan_checks.columns else "check_status"
blocking_issues_col = "blocking_check_issues" if "blocking_check_issues" in stage6_hdbscan_checks.columns else "check_issues"
if stage6_hdbscan_checks[0, blocking_status_col] != "pass":
    raise ValueError(f"Stage 6.2 hard HDBSCAN structural checks failed: {stage6_hdbscan_checks[0, blocking_issues_col]}")


### Stage 6.3: Representation and Core-Cluster Quality Evidence

Write the official Stage 6 evidence after 6.1 and 6.2 have produced aligned artifacts. Because UMAP has no PCA-style variance retention, this cell checks neighborhood retention instead: trustworthiness, kNN overlap, and sampled distance-rank correlation. For HDBSCAN, silhouette is kept as supporting core-only evidence, while density-aware DBCV is reported when available. Stage 6.4 then tests perturbation stability and cluster profile readiness; Stage 7 tests interpretability and profileability.

In [ ]:
model_suite = model_suite_from_single_candidate(
    stage6_assignment_path,
    stage6_result_path,
    stage6_core_candidate,
    umap_path=stage6_umap_path,
)
stage6_diagnostics = build_candidate_model_diagnostics(model_suite, cfg=CONFIG)
stage6_quality_path = build_stage6_representation_cluster_diagnostics(
    stage6_feature_path,
    stage6_umap_path,
    stage6_assignment_path,
    stage6_result_path,
    cfg=CONFIG,
)
stage6_quality = pl.read_csv(stage6_quality_path)
stage6_figure = plot_stage6_quality_evidence(stage6_quality_path, cfg=CONFIG)
stage6_ranked = pl.read_parquet(stage6_diagnostics["parquet"]).sort("stage6_rank")
stage6_quality_display_cols = [
    "aligned_rows",
    "source_feature_count",
    "umap_component_count",
    "umap_quality_sample_size",
    "umap_neighbor_k",
    "umap_trustworthiness",
    "umap_mean_knn_overlap_pct",
    "umap_distance_spearman",
    "cluster_count",
    "noise_pct",
    "core_coverage_pct",
    "hdbscan_dbcv_score",
    "hdbscan_dbcv_status",
    "silhouette_core_only",
    "coverage_adjusted_silhouette",
    "davies_bouldin_core_only",
    "avg_assignment_confidence",
    "hdbscan_cluster_persistence_mean",
]
stage6_quality_table = stage6_quality.select([c for c in stage6_quality_display_cols if c in stage6_quality.columns])
stage6_display_cols = [
    "stage6_rank",
    "candidate_id",
    "model_name",
    "algorithm_name",
    "assignment_policy",
    "model_variant",
    "cluster_count",
    "coverage_adjusted_silhouette",
    "silhouette",
    "davies_bouldin",
    "noise_pct",
    "core_coverage_pct",
    "soft_assigned_pct",
    "passes_quality_gate",
]
stage6_table = stage6_ranked.select([c for c in stage6_display_cols if c in stage6_ranked.columns])
stage6_best = stage6_ranked.row(0, named=True)
stage6_quality_best = stage6_quality.row(0, named=True)
stage6_report_metrics = [
    ("cluster_count", stage6_quality_best.get("cluster_count"), "10-15 client hypothesis; not a hard constraint"),
    ("noise_pct", stage6_quality_best.get("noise_pct"), "<= 60% quality gate; <= 40% strong"),
    ("core_coverage_pct", stage6_quality_best.get("core_coverage_pct"), ">= 40% useful; >= 60% strong"),
    ("quality_gate_status", stage6_hdbscan_checks[0, "quality_gate_status"] if "quality_gate_status" in stage6_hdbscan_checks.columns else stage6_best.get("passes_quality_gate"), "pass"),
    ("quality_gate_issues", stage6_hdbscan_checks[0, "quality_gate_issues"] if "quality_gate_issues" in stage6_hdbscan_checks.columns else stage6_best.get("quality_gate_reason"), "pass / None"),
    ("umap_trustworthiness", stage6_quality_best.get("umap_trustworthiness"), ">= 0.95 strong; >= 0.90 usable"),
    ("umap_mean_knn_overlap_pct", stage6_quality_best.get("umap_mean_knn_overlap_pct"), ">= 25% useful; >= 15% review"),
    ("umap_distance_spearman", stage6_quality_best.get("umap_distance_spearman"), ">= 0.70 strong; >= 0.60 usable"),
    ("hdbscan_dbcv_score", stage6_quality_best.get("hdbscan_dbcv_score"), ">= 0.25 strong; >= 0.10 usable; near 0 weak"),
    ("hdbscan_dbcv_status", stage6_quality_best.get("hdbscan_dbcv_status"), "pass"),
    ("silhouette_core_only", stage6_quality_best.get("silhouette_core_only"), ">= 0.40 strong; >= 0.25 usable"),
    ("coverage_adjusted_silhouette", stage6_quality_best.get("coverage_adjusted_silhouette"), ">= 0.25 strong; >= 0.15 usable"),
    ("davies_bouldin_core_only", stage6_quality_best.get("davies_bouldin_core_only"), "lower is better; < 1.0 usually usable"),
    ("avg_assignment_confidence", stage6_quality_best.get("avg_assignment_confidence"), ">= 0.40 stronger; >= 0.30 review"),
    ("assignment_policy", stage6_best.get("assignment_policy"), "hard_hdbscan_core_noise_retained"),
    ("soft_assigned_pct", stage6_best.get("soft_assigned_pct"), "0% for official hard-core recipe"),
]
stage6_report = write_stage_report(
    "06",
    "Hard UMAP-HDBSCAN Core Tribe Discovery",
    summary=[
        f"Selection feature set: {selection_feature_set} (official signal is product identity plus quantity, with no spend or demographic inputs).",
        "Stage 6.1 builds the UMAP customer manifold; Stage 6.2 runs hard HDBSCAN with noise retained.",
        "Stage 6.3 reports UMAP neighborhood retention and density-aware HDBSCAN validity evidence for the official recipe.",
        f"Official core model: {stage6_best['model_name']} / {stage6_best['model_variant']}",
    ],
    metrics=stage6_report_metrics,
    figures={
        "Stage 6.1 UMAP representation": stage6_umap_figure,
        "Stage 6.2 hard HDBSCAN assignment map": stage6_assignment_figure,
        "Stage 6.3 representation and density evidence": stage6_figure,
    },
    artifacts={
        "UMAP representation": model_suite.get("umap_path"),
        "Stage 6.1 UMAP checks": stage6_umap_checks_path,
        "Hard HDBSCAN assignments": stage6_assignment_path,
        "Stage 6.2 HDBSCAN checks": stage6_hdbscan_checks_path,
        "Stage 6.3 representation and cluster quality CSV": stage6_quality_path,
        "Diagnostics parquet": stage6_diagnostics["parquet"],
        "Diagnostics summary CSV": stage6_diagnostics["summary_csv"],
    },
    cfg=CONFIG,
    max_metric_rows=24,
)
display_stage_report(stage6_report)
display(Image(filename=str(stage6_figure)))
display(stage6_quality_table)
display(stage6_table.head(12))
display(stage6_assignment_source_table)


<a id="stage-6-4"></a>

## Stage 6.4: Cluster Stability and Profile Readiness

Stage 6.4 is the handoff from clustering into profiling. Stage 6.3 already checks representation quality and HDBSCAN density validity; this step asks whether the retained clusters themselves are stable enough and confidence-backed enough to interpret.

Use this stage to answer: which clusters are ready for product-lift profiling, and which clusters should be handled cautiously because they are small, low-confidence, or sensitive to perturbation?

The notebook-facing summary is written to `outputs/<mode>/reports/stage_06_4.md`, with cluster readiness, stability diagnostics, and winner projection figures displayed directly below.


In [ ]:
from IPython.display import Image, display
import polars as pl

from src.cluster_validation import build_cluster_validity_stability_report
from src.stage_reports import display_stage_report, write_stage_report
from src.visualization import build_2d_projection_figures, plot_stage6_cluster_readiness

cluster_stability = build_cluster_validity_stability_report(
    model_suite,
    feature_sets[selection_feature_set],
    output_path=CONFIG.artifacts / "stage6" / "stage6_4_cluster_stability_readiness.parquet",
    cfg=CONFIG,
)

stability_table = pl.read_csv(cluster_stability["summary_csv"]).select([
    "candidate_id",
    "cluster_count",
    "noise_pct",
    "cluster_size_cv",
    "jitter_ari_mean",
    "jitter_ari_std",
    "jitter_label_recovery_accuracy_mean",
    "validity_note",
])
stage6_winner_key = pl.read_parquet(stage6_diagnostics["parquet"]).sort("stage6_rank")[0, "candidate_id"]
stage6_4_winner_stability = stability_table.filter(pl.col("candidate_id") == stage6_winner_key)
cluster_readiness_table = pl.read_csv(cluster_stability["cluster_summary_csv"])
stage6_4_cluster_readiness = cluster_readiness_table.filter(pl.col("candidate_id") == stage6_winner_key)

stage6_4_projection_dir = CONFIG.figures
stage6_4_projection_figures = build_2d_projection_figures(
    feature_sets[selection_feature_set],
    model_suite["assignment_paths"][stage6_winner_key],
    output_dir=stage6_4_projection_dir,
    output_prefix="stage_06_4_winner_projection",
    stage_label="Stage 6.4 figures",
    cfg=CONFIG,
)
stage6_4_cluster_readiness_figure = plot_stage6_cluster_readiness(
    cluster_stability["cluster_summary_csv"],
    cfg=CONFIG,
)
stage6_4_metrics_row = stage6_4_winner_stability.row(0, named=True)
stage6_4_report_metrics = [
    ("cluster_count", stage6_4_metrics_row.get("cluster_count"), "10-15 client hypothesis; not a hard constraint"),
    ("noise_pct", stage6_4_metrics_row.get("noise_pct"), "<= 60% quality gate; <= 40% strong"),
    ("cluster_size_cv", stage6_4_metrics_row.get("cluster_size_cv"), "<= 1.5 gate; lower is more balanced"),
    ("jitter_ari_mean", stage6_4_metrics_row.get("jitter_ari_mean"), ">= 0.80 strong; >= 0.60 usable"),
    ("jitter_label_recovery_accuracy_mean", stage6_4_metrics_row.get("jitter_label_recovery_accuracy_mean"), ">= 0.85 strong; >= 0.70 usable"),
    ("clusters_marked_review", stage6_4_cluster_readiness.filter(pl.col("profile_readiness") == "review").height, "0 ideal; review before profiling if > 0"),
    ("clusters_marked_usable_or_strong", stage6_4_cluster_readiness.filter(pl.col("profile_readiness").is_in(["usable", "strong"])).height, "all retained clusters"),
]
stage6_4_report = write_stage_report(
    "06_4",
    "Cluster Stability and Profile Readiness",
    summary=[
        f"Stage 6 winner checked: {stage6_winner_key}",
        "Stage 6.4 focuses on the retained clusters themselves: stability, confidence, size, and readiness for profiling.",
        "This is the technical gate before Stage 7 product-lift interpretation.",
    ],
    metrics=stage6_4_report_metrics,
    figures={
        "Cluster readiness": stage6_4_cluster_readiness_figure,
        "PCA projection": stage6_4_projection_figures.get("pca"),
        "UMAP projection": stage6_4_projection_figures.get("umap"),
    },
    artifacts={
        "Stability/readiness parquet": cluster_stability["parquet"],
        "Stability/readiness summary CSV": cluster_stability["summary_csv"],
        "Cluster readiness parquet": cluster_stability["cluster_parquet"],
        "Cluster readiness summary CSV": cluster_stability["cluster_summary_csv"],
        "Projection figures root": stage6_4_projection_dir,
    },
    cfg=CONFIG,
    max_metric_rows=16,
)
display_stage_report(stage6_4_report)
display(stability_table.head(20))
display(stage6_4_winner_stability)
display(stage6_4_cluster_readiness.select([
    "tribe_id",
    "customers",
    "customer_share_pct",
    "mean_assignment_confidence",
    "p10_assignment_confidence",
    "jitter_label_recovery_accuracy_mean",
    "profile_readiness",
    "readiness_issues",
]).sort("tribe_id"))
display(Image(filename=str(stage6_4_cluster_readiness_figure)))
if "umap" in stage6_4_projection_figures:
    display(Image(filename=str(stage6_4_projection_figures["umap"])))
display(Image(filename=str(stage6_4_projection_figures["pca"])))


<a id="stage-7"></a>

## Stage 7: Deep Tribe Profiling and Interpretation

Stage 7 is the unpacking stage. The clustering algorithm found dense product-purchase groups; this stage turns that output into a readable evidence storyline and keeps uncertainty visible. The goal is not more artifacts for their own sake. The goal is a cohesive argument for each tribe: why it exists, what makes it different, what supports the read, and what should remain caveated.

Read Stage 7 as an evidence ladder:

| Step | Question | Evidence |
|---|---|---|
| 1. Selected solution | Which clustering result are we explaining? | Stage 6 ranked winner and hard HDBSCAN assignment |
| 2. Technical trust | Are the tribes stable enough to interpret? | Stage 6.4 readiness, confidence, perturbation checks |
| 3. Distinctive behavior | What makes each tribe different from the rest? | Product/category lift, lift-vs-rest, q-values, customer coverage |
| 4. Supporting context | What reinforces or nuances the read? | Product terms, themes, sectors, customer-context ANOVA |
| 5. Action overlays | Where can the story become useful? | Evidence-backed subsegments from lifted terms/themes |
| 6. Noise audit | What is hiding outside the core tribes? | Noise share, behavior contrasts, product/theme/term over-indexing, recommended action |
| 7. Caveats | What must not be overstated? | Readiness issues, label confidence, noise customers, guardrails |

The notebook display is intentionally compact: the Stage 7 report, the evidence storyline table, and the main evidence dashboard. Detailed product tables, all-tribe comparison files, atlas, dossier, and LLM prompt pack are written as supporting proof. Open them when you need to defend a specific claim or investigate a caveat.

Working tribe names are deterministic review labels, not discovered personas. The optional LLM layer receives aggregate evidence only and is explicitly constrained from demographic, household, health, income, age, gender, nationality, religion, or identity inference.


In [ ]:
import polars as pl
from IPython.display import Image, Markdown, display

from src.profiling import (
    customer_metric_anova_table,
    flatten_profiles_for_csv,
    profile_evidence_metrics_table,
    profile_overview_table,
    profile_quality_summary,
    profile_readiness_evidence_table,
    profile_subsegment_opportunity_table,
    stage7_storyline_table,
    profile_tribes,
    tribe_comparison_table,
    tribe_product_summary_tables,
    write_campaign_signal_artifacts,
    write_clustering_atlas_artifacts,
    write_cluster_summary_artifacts,
    write_customer_subsegment_artifacts,
    write_discovered_product_term_artifacts,
    write_llm_profile_interpretation_pack,
    write_noise_audit_artifacts,
    write_stage7_storyline_artifacts,
    write_tribe_comparison_artifacts,
    write_tribe_product_summary_artifacts,
    write_tribe_profile_report,
)
from src.stage_reports import display_stage_report, write_stage_report
from src.visualization import (
    plot_top_lifts,
    plot_tribe_profile_comparison_heatmap,
    plot_tribe_theme_lift_heatmap,
    plot_tribe_vs_population_evidence_dashboard,
)

stage6_ranked = pl.read_parquet(stage6_diagnostics["parquet"]).sort("stage6_rank")
stage6_winner = stage6_ranked.row(0, named=True)
selected_key = stage6_winner["candidate_id"]
selected_assignment_path = model_suite["assignment_paths"][selected_key]
selected_profile_path = profile_tribes(
    selected_assignment_path,
    transactions=transactions,
    behavior_path=behavior_path,
    cfg=CONFIG,
)
profile_paths = {selected_key: selected_profile_path}

stage7_artifacts_dir = CONFIG.artifacts / "stage7"
stage7_evidence_dir = stage7_artifacts_dir / "deep_profile_evidence"
stage7_figures_dir = CONFIG.figures
stage7_lift_dir = stage7_figures_dir / "tribe_lifts"
for path in [stage7_artifacts_dir, stage7_evidence_dir, stage7_figures_dir, stage7_lift_dir]:
    path.mkdir(parents=True, exist_ok=True)

stage7_quality = profile_quality_summary(selected_profile_path, cfg=CONFIG)
stage7_readiness_path = stage7_artifacts_dir / "stage7_profile_readiness_evidence.csv"
stage7_readiness = profile_readiness_evidence_table(
    selected_profile_path,
    cluster_readiness_path=cluster_stability.get("cluster_summary_csv") if "cluster_stability" in globals() else None,
    output_csv=stage7_readiness_path,
    cfg=CONFIG,
)
stage7_ready_clusters = stage7_readiness.filter(pl.col("profiling_readiness").is_in(["ready", "ready_strong"])).height
stage7_review_clusters = stage7_readiness.filter(pl.col("profiling_readiness") == "review").height

profile_export = flatten_profiles_for_csv(
    selected_profile_path,
    stage7_evidence_dir / f"stage7_tribe_profiles_flat_{CONFIG.mode}.csv",
)
stage7_cluster_summary_paths = write_cluster_summary_artifacts(
    selected_profile_path,
    output_csv=stage7_artifacts_dir / f"stage7_core_tribe_summary_{CONFIG.mode}.csv",
    cfg=CONFIG,
)
product_summary_paths = write_tribe_product_summary_artifacts(
    selected_profile_path,
    output_dir=stage7_artifacts_dir / "tribe_product_summaries",
    combined_output_csv=stage7_artifacts_dir / f"stage7_tribe_product_summary_long_{CONFIG.mode}.csv",
    cfg=CONFIG,
)
tribe_product_tables = tribe_product_summary_tables(selected_profile_path, cfg=CONFIG)
tribe_comparison_artifacts = write_tribe_comparison_artifacts(
    selected_profile_path,
    output_csv=stage7_artifacts_dir / f"stage7_tribe_comparison_{CONFIG.mode}.csv",
    output_md=stage7_artifacts_dir / f"stage7_tribe_comparison_{CONFIG.mode}.md",
    output_html=stage7_artifacts_dir / f"stage7_tribe_comparison_{CONFIG.mode}.html",
    readiness_path=stage7_readiness_path,
    cfg=CONFIG,
)
tribe_comparison = tribe_comparison_table(selected_profile_path, readiness_path=stage7_readiness_path, cfg=CONFIG)
stage7_profile_overview = profile_overview_table(selected_profile_path, cfg=CONFIG)
stage7_profile_evidence = profile_evidence_metrics_table(selected_profile_path, cfg=CONFIG)

customer_metric_tests_path = stage7_artifacts_dir / f"stage7_customer_metric_anova_{CONFIG.mode}.csv"
customer_metric_tests = customer_metric_anova_table(
    selected_assignment_path,
    behavior_path=behavior_path,
    output_csv=customer_metric_tests_path,
    cfg=CONFIG,
)
q_threshold = float(CONFIG.get("profiling.significance_q_threshold", 0.05))
stage7_significant_customer_metric_tests = (
    customer_metric_tests.filter(pl.col("anova_q_value") <= q_threshold).height
    if not customer_metric_tests.is_empty() and "anova_q_value" in customer_metric_tests.columns
    else 0
)

stage7_noise_audit_paths = write_noise_audit_artifacts(
    selected_assignment_path,
    transactions=transactions,
    behavior_path=behavior_path,
    output_csv=stage7_artifacts_dir / f"stage7_noise_audit_{CONFIG.mode}.csv",
    output_md=stage7_artifacts_dir / f"stage7_noise_audit_{CONFIG.mode}.md",
    output_html=stage7_artifacts_dir / f"stage7_noise_audit_{CONFIG.mode}.html",
    cfg=CONFIG,
)
stage7_noise_audit = pl.read_csv(stage7_noise_audit_paths["csv"])
stage7_noise_summary = stage7_noise_audit.filter(pl.col("evidence_type") == "overall_recommendation")
stage7_noise_summary_row = stage7_noise_summary.row(0, named=True) if stage7_noise_summary.height else {}

subsegment_opportunities = profile_subsegment_opportunity_table(selected_profile_path, cfg=CONFIG)
subsegment_opportunities_path = stage7_artifacts_dir / f"stage7_subsegment_opportunities_{CONFIG.mode}.csv"
subsegment_opportunities.write_csv(subsegment_opportunities_path)
customer_subsegment_paths = write_customer_subsegment_artifacts(
    selected_assignment_path,
    selected_profile_path,
    transactions=transactions,
    output_parquet=stage7_evidence_dir / f"stage7_customer_subsegment_tags_{CONFIG.mode}.parquet",
    output_summary_csv=stage7_artifacts_dir / f"stage7_customer_subsegment_summary_{CONFIG.mode}.csv",
    output_summary_md=stage7_artifacts_dir / f"stage7_customer_subsegment_summary_{CONFIG.mode}.md",
    output_summary_html=stage7_artifacts_dir / f"stage7_customer_subsegment_summary_{CONFIG.mode}.html",
    cfg=CONFIG,
)
discovered_term_paths = write_discovered_product_term_artifacts(
    selected_profile_path,
    output_csv=stage7_artifacts_dir / f"stage7_discovered_product_terms_{CONFIG.mode}.csv",
    output_md=stage7_artifacts_dir / f"stage7_discovered_product_terms_{CONFIG.mode}.md",
    output_html=stage7_artifacts_dir / f"stage7_discovered_product_terms_{CONFIG.mode}.html",
    cfg=CONFIG,
)
campaign_signal_paths = write_campaign_signal_artifacts(
    selected_profile_path,
    output_csv=stage7_artifacts_dir / f"stage7_campaign_signal_overlays_{CONFIG.mode}.csv",
    output_md=stage7_artifacts_dir / f"stage7_campaign_signal_overlays_{CONFIG.mode}.md",
    output_html=stage7_artifacts_dir / f"stage7_campaign_signal_overlays_{CONFIG.mode}.html",
    cfg=CONFIG,
)

stage7_storyline_paths = write_stage7_storyline_artifacts(
    selected_profile_path,
    readiness_path=stage7_readiness_path,
    comparison_path=tribe_comparison_artifacts["csv"],
    subsegment_summary_path=customer_subsegment_paths["summary_csv"],
    customer_metric_tests_path=customer_metric_tests_path,
    output_csv=stage7_artifacts_dir / f"stage7_evidence_storyline_{CONFIG.mode}.csv",
    output_md=stage7_artifacts_dir / f"stage7_evidence_storyline_{CONFIG.mode}.md",
    output_html=stage7_artifacts_dir / f"stage7_evidence_storyline_{CONFIG.mode}.html",
    cfg=CONFIG,
)
stage7_storyline = stage7_storyline_table(
    selected_profile_path,
    readiness_path=stage7_readiness_path,
    comparison_path=tribe_comparison_artifacts["csv"],
    subsegment_summary_path=customer_subsegment_paths["summary_csv"],
    cfg=CONFIG,
)
profile_report_path = write_tribe_profile_report(
    selected_profile_path,
    output_path=stage7_artifacts_dir / f"stage7_deep_tribe_dossiers_{CONFIG.mode}.md",
    cfg=CONFIG,
)

tribe_vs_population_dashboard = plot_tribe_vs_population_evidence_dashboard(
    selected_profile_path,
    output_path=stage7_figures_dir / f"stage_07_tribe_vs_population_evidence_dashboard_{CONFIG.mode}.png",
    cfg=CONFIG,
)
profile_comparison_heatmap = plot_tribe_profile_comparison_heatmap(
    selected_profile_path,
    output_path=stage7_figures_dir / f"stage_07_tribe_profile_comparison_heatmap_{CONFIG.mode}.png",
    cfg=CONFIG,
)
stage7_theme_lift_heatmap = plot_tribe_theme_lift_heatmap(
    selected_profile_path,
    output_path=stage7_figures_dir / f"stage_07_tribe_theme_lift_heatmap_{CONFIG.mode}.png",
    cfg=CONFIG,
)
top_lift_paths = plot_top_lifts(
    selected_profile_path,
    output_dir=stage7_lift_dir,
    cfg=CONFIG,
)

clustering_atlas_paths = write_clustering_atlas_artifacts(
    selected_profile_path,
    subsegment_summary_path=customer_subsegment_paths["summary_csv"],
    figure_paths={
        "tribe_vs_population_dashboard": tribe_vs_population_dashboard,
        "profile_comparison_heatmap": profile_comparison_heatmap,
        "theme_lift_heatmap": stage7_theme_lift_heatmap,
        "top_lift_plots": top_lift_paths[:5],
    },
    output_csv=stage7_artifacts_dir / f"stage7_clustering_atlas_{CONFIG.mode}.csv",
    output_md=stage7_artifacts_dir / f"stage7_clustering_atlas_{CONFIG.mode}.md",
    output_html=stage7_artifacts_dir / f"stage7_clustering_atlas_{CONFIG.mode}.html",
    cfg=CONFIG,
)
llm_prompt_paths = write_llm_profile_interpretation_pack(
    selected_profile_path,
    readiness_path=stage7_readiness_path,
    comparison_path=tribe_comparison_artifacts["csv"],
    output_json=stage7_artifacts_dir / f"stage7_llm_interpretation_pack_{CONFIG.mode}.json",
    output_md=stage7_artifacts_dir / f"stage7_llm_interpretation_pack_{CONFIG.mode}.md",
    cfg=CONFIG,
)

stage7_model_table = pl.DataFrame([
    {
        "stage6_rank": stage6_winner.get("stage6_rank"),
        "candidate_id": selected_key,
        "cluster_count": stage6_winner.get("cluster_count"),
        "profile_path": str(selected_profile_path),
        "storyline": str(stage7_storyline_paths["markdown"]),
        "noise_audit": str(stage7_noise_audit_paths["markdown"]),
        "llm_prompt_pack": str(llm_prompt_paths["markdown"]),
    }
])
stage7_report = write_stage_report(
    "07",
    "Deep Tribe Profiling and Interpretation",
    summary=[
        f"Profiled selected Stage 6 model: {selected_key}",
        "Stage 7 is read as an evidence ladder: readiness, product distinctiveness, supporting context, subsegments, and caveats.",
        "The primary handoff artifact is the evidence storyline; detailed files remain available as supporting proof.",
        "The HDBSCAN noise population is audited separately before any soft-assignment or second-pass-clustering decision.",
        "Spend/KPIs are used only for post-clustering customer understanding and statistical profiling.",
        "The LLM prompt pack is aggregate-only and includes explicit guardrails against demographic or identity inference.",
    ],
    metrics=[
        ("storyline_rows", stage7_storyline.height, "one evidence-story row per retained tribe"),
        ("profiled_clusters", stage7_quality.get("profiled_clusters"), "matches retained Stage 6 tribes"),
        ("per_tribe_product_tables", len(tribe_product_tables), "one table per retained tribe"),
        ("profile_ready_clusters", stage7_ready_clusters, "all retained tribes ideal"),
        ("profile_review_clusters", stage7_review_clusters, "0 ideal; review before naming externally"),
        ("clusters_with_significant_product_lift", stage7_quality.get("clusters_with_significant_product_lift"), "most tribes; q<=0.05"),
        ("avg_significant_product_lifts_per_cluster", stage7_quality.get("avg_significant_product_lifts_per_cluster"), ">=1.0 preferred"),
        ("customer_metric_anova_tests", customer_metric_tests.height, "customer-level profiling metrics tested across tribes"),
        ("significant_customer_metric_anova_tests", stage7_significant_customer_metric_tests, "q<=0.05 after FDR"),
        ("noise_customers", stage7_noise_summary_row.get("noise_observations"), "kept outside official core tribes"),
        ("noise_share_pct", stage7_noise_summary_row.get("noise_share_pct"), "drives keep/revisit/second-pass recommendation"),
        ("noise_audit_signals", stage7_noise_summary_row.get("evidence_strength"), "hidden structure signals found inside noise"),
        ("noise_recommended_action", stage7_noise_summary_row.get("recommended_action"), "do not force soft assignment without this review"),
        ("subsegment_opportunity_rows", subsegment_opportunities.height, "evidence-backed within-tribe overlays"),
        ("customer_subsegment_rows", pl.read_csv(customer_subsegment_paths["summary_csv"]).height, "theme/term subsegment summaries"),
        ("avg_max_product_lift_vs_rest", stage7_quality.get("avg_max_product_lift_vs_rest"), ">1.5 indicates clear over-indexing"),
        ("avg_soft_assigned_share", stage7_quality.get("avg_soft_assigned_share"), "0 for hard-core official model"),
        ("llm_prompted_tribes", stage7_quality.get("profiled_clusters"), "aggregate evidence only; no customer IDs"),
    ],
    figures={
        "Tribe vs population dashboard": tribe_vs_population_dashboard,
        "Between-tribe profile comparison heatmap": profile_comparison_heatmap,
        "Theme lift heatmap": stage7_theme_lift_heatmap,
        "Top product lift plot directory": stage7_lift_dir,
    },
    artifacts={
        "Evidence storyline MD": stage7_storyline_paths["markdown"],
        "Evidence storyline HTML": stage7_storyline_paths["html"],
        "Selected profile parquet": selected_profile_path,
        "Flat profile CSV": profile_export,
        "Core tribe summary": stage7_cluster_summary_paths["csv"],
        "Per-tribe product summary directory": product_summary_paths["directory"],
        "Product summary long table": product_summary_paths["combined_csv"],
        "All-tribe comparison CSV": tribe_comparison_artifacts["csv"],
        "All-tribe comparison HTML": tribe_comparison_artifacts["html"],
        "Customer metric ANOVA tests": customer_metric_tests_path,
        "Noise population audit": stage7_noise_audit_paths["markdown"],
        "Profile readiness evidence": stage7_readiness_path,
        "Subsegment opportunities": subsegment_opportunities_path,
        "Customer subsegment summary": customer_subsegment_paths["summary_csv"],
        "Discovered product terms": discovered_term_paths["csv"],
        "Campaign signal overlays": campaign_signal_paths["csv"],
        "Deep tribe dossier": profile_report_path,
        "Clustering atlas HTML": clustering_atlas_paths["html"],
        "LLM interpretation prompt pack": llm_prompt_paths["markdown"],
    },
    cfg=CONFIG,
    max_metric_rows=18,
)

display_stage_report(stage7_report, max_lines=80)
display(stage7_model_table)
display(Markdown(
    f"### Stage 7 Evidence Storyline\n\n"
    f"Primary storyline: `{stage7_storyline_paths['markdown']}`  \n"
    f"Readable HTML: `{stage7_storyline_paths['html']}`  \n"
    "Use this as the first reading layer. Open the detailed artifacts only to support a claim or inspect a caveat."
))
display(stage7_storyline)
display(Image(filename=str(tribe_vs_population_dashboard)))
display(Markdown(
    "### Supporting Evidence Index\n\n"
    f"- All-tribe comparison: `{tribe_comparison_artifacts['html']}`\n"
    f"- Deep tribe dossier: `{profile_report_path}`\n"
    f"- Clustering atlas: `{clustering_atlas_paths['html']}`\n"
    f"- Product summary directory: `{product_summary_paths['directory']}`\n"
    f"- Customer-context ANOVA: `{customer_metric_tests_path}`\n"
    f"- Noise audit: `{stage7_noise_audit_paths['html']}`\n"
    f"- Subsegment summary: `{customer_subsegment_paths['summary_html']}`\n"
    f"- LLM prompt pack: `{llm_prompt_paths['markdown']}`"
))

selected_profile_path
